In [0]:
from pyspark.sql import functions as F

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

gold_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_revenue_incremental"
)

checkpoint_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_checkpoint"
)

gold_state = spark.table(gold_table).first()

gold_version = gold_state["last_applied_version"]

latest_version = (
    spark.sql(f"DESCRIBE HISTORY {silver_table}")
    .agg(F.max("version").alias("latest_version"))
    .first()["latest_version"]
)

print("==== INCREMENTAL GOLD PIPELINE====")
print("Gold transactions:", gold_state["TotalTransactions"])
print("Gold revenue:", gold_state["TotalNetRevenue"])
print("Gold checkpoint:",gold_version)
print("Latest Silver version:", latest_version)

if latest_version > gold_version:
    print("New CHANGES AVAILABLE")
else:
    print("ALREADY UP TO DATE")

In [0]:

from pyspark.sql import functions as F
from decimal import Decimal

# Read the current Gold processing version
gold_state = spark.table(gold_table).first()
gold_version = gold_state["last_applied_version"]

# Find the latest Silver Delta version
latest_version = (
    spark.sql(f"DESCRIBE HISTORY {silver_table}")
    .agg(F.max("version").alias("latest_version"))
    .first()["latest_version"]
)

if latest_version <= gold_version:
    print("SKIPPED: Gold is already up to date.")

else:
    print(
        f"Processing Silver versions "
        f"{gold_version + 1} to {latest_version}"
    )

    # Read new Silver changes
    df_changes = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", gold_version + 1)
        .option("endingVersion", latest_version)
        .table(silver_table)
    )

    # Calculate signed revenue and transaction adjustments
    df_adjustments = (
        df_changes
        .withColumn(
            "line_total",
            F.round(
                F.col("Quantity").cast("decimal(18,2)") *
                F.col("UnitPrice").cast("decimal(18,2)"),
                2
            )
        )
        .withColumn(
            "revenue_delta",
            F.when(
                F.col("_change_type").isin(
                    "insert", "update_postimage"
                ),
                F.col("line_total")
            ).otherwise(-F.col("line_total"))
        )
        .withColumn(
            "count_delta",
            F.when(F.col("_change_type") == "insert", 1)
            .when(F.col("_change_type") == "delete", -1)
            .otherwise(0)
        )
    )

    totals = df_adjustments.agg(
        F.sum("count_delta").alias("count_delta"),
        F.sum("revenue_delta").alias("revenue_delta")
    ).first()

    count_delta = int(totals["count_delta"] or 0)
    revenue_delta = totals["revenue_delta"] or Decimal("0.00")

    print("Transaction adjustment:", count_delta)
    print("Revenue adjustment:", revenue_delta)

    # Create a one-row source for the Delta MERGE
    spark.createDataFrame(
        [(
            count_delta,
            str(revenue_delta),
            int(latest_version),
            int(gold_version)
        )],
        [
            "count_delta",
            "revenue_delta_text",
            "target_version",
            "expected_version"
        ]
    ).createOrReplaceTempView("gold_cdf_adjustment")

    # Atomically update Gold totals and its processing version
    spark.sql(f"""
        MERGE INTO {gold_table} AS target
        USING (
            SELECT
                count_delta,
                CAST(revenue_delta_text AS DECIMAL(38,2))
                    AS revenue_delta,
                target_version,
                expected_version
            FROM gold_cdf_adjustment
        ) AS source
        ON target.last_applied_version = source.expected_version
        WHEN MATCHED THEN UPDATE SET
            target.TotalTransactions =
                target.TotalTransactions + source.count_delta,
            target.TotalNetRevenue =
                target.TotalNetRevenue + source.revenue_delta,
            target.last_applied_version = source.target_version
    """)

    print("Gold MERGE completed.")

display(spark.table(gold_table))


In [0]:

# Step 3: Synchronize Gold checkpoint

# Read the latest successfully applied Gold version
gold_state = spark.table(gold_table).first()

applied_version = gold_state["last_applied_version"]

# Update checkpoint only when it is behind Gold
spark.sql(f"""
    UPDATE {checkpoint_table}
    SET
        last_processed_version = {int(applied_version)},
        updated_at = current_timestamp()
    WHERE pipeline_name = 'silver_to_gold_revenue'
      AND last_processed_version < {int(applied_version)}
""")

# Verify the checkpoint
checkpoint_state = spark.sql(f"""
    SELECT last_processed_version
    FROM {checkpoint_table}
    WHERE pipeline_name = 'silver_to_gold_revenue'
""").first()

assert checkpoint_state is not None, "Checkpoint record missing"

assert (
    checkpoint_state["last_processed_version"] == applied_version
), "Gold checkpoint mismatch"

print("===== GOLD CHECKPOINT VALIDATION =====")
print("Gold applied version:", applied_version)
print(
    "Tracking checkpoint:",
    checkpoint_state["last_processed_version"]
)
print("PASS: Gold checkpoint synchronized.")


In [0]:

from pyspark.sql import functions as F

source_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_source_snapshot"
)

test_id = "adf_e2e_test_001"

# Check whether this test record already exists
existing = (
    spark.table(source_table)
    .filter(F.col("test_record_id") == test_id)
    .count()
)

if existing == 0:
    new_record = spark.createDataFrame(
        [(
            "ADF-TEST-001",
            "TEST-002",
            "ADF End-to-End Test Item",
            "3",
            "2026-10-26 10:00:00",
            "20.00",
            "TEST-CUSTOMER-002",
            "Canada",
            "adf_e2e_test",
            test_id
        )],
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country",
            "source_file",
            "test_record_id"
        ]
    )

    (
        new_record.write
        .format("delta")
        .mode("append")
        .saveAsTable(source_table)
    )

    print("New source test record inserted.")
else:
    print("Test record already exists.")

print("Source records:", spark.table(source_table).count())


In [0]:

from pyspark.sql import functions as F

source_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_source_snapshot"
)

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

test_id = "adf_e2e_test_001"

df_test = (
    spark.table(source_table)
    .filter(F.col("test_record_id") == test_id)
    .withColumn(
        "batch_partition",
        F.pmod(F.xxhash64("test_record_id"), F.lit(2))
    )
)

display(
    df_test.select(
        "test_record_id",
        "Quantity",
        "UnitPrice",
        "batch_partition"
    )
)

silver_matches = (
    spark.table(silver_table)
    .filter(F.col("test_record_id") == test_id)
    .count()
)

print("Already in Silver:", silver_matches)


In [0]:
silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

test_id = "adf_e2e_test_001"

silver_count = (
    spark.table(silver_table)
    .filter(f"test_record_id = '{test_id}'")
    .count()
)

print("Already in Silver:", silver_count)